# 02 — Deploy pipelines

Wealth ML SageMaker Foundation (WSF).

Use the **output of notebook 01**, not its metadata CSV.

| Mode | Data input here | What the pipeline's GetData step does |
|---|---|---|
| csv_direct | Generated sample.csv (or another ready flat CSV) | Reads the mounted CSV and copies it unchanged to raw/data.csv |
| feature_store | Feature Store configuration URI saved by notebook 01 | Queries the existing store and writes raw/data.csv |

The feature mapping is used for Feature Store ingestion/query configuration;
CSV-direct does not require it. Both routes hand a flat CSV to FeatureSelection.
The modes share an output contract, not necessarily identical column sets:
the current Feature Store query selects configured model features, while
CSV-direct preserves all supplied columns.

Both notebooks read **notebooks/config.json**. Notebook 01 saves the generated data input there.
Run All here after notebook 01; no copy/paste between notebooks is needed. Optional settings are in the **Defaults** cell below.
For an existing CSV or Feature Store binding, set the corresponding config value and run this notebook directly.
This notebook publishes **one shared training pipeline and one shared inference pipeline**.
Adding or renaming a model folder updates their ModelId dropdown when republished.
The default names are **wsf-dev-training** and **wsf-dev-inference**.
They use `wsf-dev/` in S3 and leave the existing `source-dev` pipelines and outputs in place.
It does not create one pipeline pair per model.

Use a fresh kernel for this checkout; an older checkout can otherwise remain imported.

Model Registry is required. The platform must create a group named
**`<prefix><ModelId>-sm-<environment>`** for each selected model before publication. Set registry_group_prefix in config.json, including its trailing separator.
An empty prefix preserves the original group names.
Training registers a PendingManualApproval version after the quality gate passes.
Approve a version, then start inference with the same **ModelId**. Inference resolves
the highest Completed, Approved version and uses its saved model and image.
Registry access errors or no approved version stop the pipeline; there is no S3 bypass.
Keep `include_hpo=False` for now; HPO steps and tuning inputs remain hidden.
Inference defaults to **Processing with uploaded pipeline code** and compute caching.
Run All to publish your current inference files. If a model version is already
approved, start inference directly afterwards; there is no need to rerun training.
The registered model and its saved metadata are still used.


## Read your shared inputs

- Read bucket, pipeline role, image and registry_group_prefix from notebooks/config.json.
- Read csv_input or feature_store_config_uri saved there by notebook 01; nothing resets those paths.
- If the file is missing, create a blank template and stop for you to supply values.
- Keep optional settings in the Defaults cell below; target/key/task settings remain in parameters.json.


In [ ]:
from pathlib import Path
import json

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "mlops-models/src").is_dir())
CONFIG_PATH = ROOT / "notebooks/config.json"
TEMPLATE_PATH = ROOT / "notebooks/config.example.json"
if not CONFIG_PATH.exists():
    CONFIG_PATH.write_text(TEMPLATE_PATH.read_text())
    raise ValueError(f"Created {CONFIG_PATH}. Fill in your inputs in that file, then Run All again. No notebook cell edits are needed.")

# Read supplied inputs; optional settings are declared in the Defaults cell below.
notebook_config = json.loads(TEMPLATE_PATH.read_text()) | json.loads(CONFIG_PATH.read_text())
print("Notebook configuration:", CONFIG_PATH)
bucket = notebook_config["bucket"]
role_arn = notebook_config["role_arn"]
image_uri = notebook_config["image_uri"]
csv_input = notebook_config["csv_input"]
feature_store_config_uri = notebook_config["feature_store_config_uri"]
registry_group_prefix = notebook_config["registry_group_prefix"]


## Defaults — deployment

- Declare namespace, data mode, Registry environment and optional job settings here.
- Keep data_source and namespace consistent with the Defaults cell in notebook 01.
- include_hpo=False hides HPO; cache_steps=True enables eligible compute caching.
- model_code_source="pipeline" uses uploaded inference code with the approved saved model; inputs remain in config.json.


In [ ]:
namespace = "wsf-dev"
data_source = "csv_direct"
environment = "dev"
deploy_name = "my-review"
model_ids = None
include_hpo = False
cache_steps = True
model_code_source = "pipeline"
universe_data_root = ""
job_role_arn = role_arn  # Set a different job role here if required.
batch_runtime = "processing"
infrastructure = {
    "network": {"isolation": False, "subnet_ids": [], "security_group_ids": []},
    "output_kms_key_arn": "",
    "volume_kms_key_arn": "",
}
csv_preflight_ok = csv_upload_verified = False
print("Input mode:", data_source, "| cache_steps:", cache_steps)
print("Inference runtime:", batch_runtime, "| code source:", model_code_source)


## Validate inputs and source

- Show target_key, primary_key and task_type from every selected use case's parameters.json.
- Verify that the requested CSV exists, has unique headers and contains rows; preview it and record its checksum.
- Report target presence without requiring one CSV to contain every use case's target; choose ModelId and its data at execution.
- Check Python syntax and show the checkout, model folders and output root before publishing.


In [ ]:
from uuid import uuid4
import csv
import hashlib
import io
import sys
import boto3
import pandas as pd

csv_preflight_ok = csv_upload_verified = False
bucket = bucket.strip().removeprefix("s3://").rstrip("/")
role_arn, image_uri = role_arn.strip(), image_uri.strip()
csv_input = csv_input.strip()
print("Effective csv_input:", repr(csv_input))
print("Current notebook directory:", Path.cwd())
print("cache_steps:", cache_steps)
assert bucket and "/" not in bucket and role_arn and image_uri, f"STOP: fill in bucket, role_arn and image_uri in {CONFIG_PATH}."
assert data_source in {"feature_store", "csv_direct"}, "STOP: choose csv_direct or feature_store."
if data_source == "feature_store":
    assert feature_store_config_uri, f"STOP: supply feature_store_config_uri in {CONFIG_PATH}, or complete notebook 01 in feature_store mode."
else:
    assert csv_input, f"STOP: csv_input is empty in {CONFIG_PATH}. Run notebook 01, or set an existing CSV path in that file. Nothing will be published."

sys.path[:0] = [str(ROOT / "mlops-models"), str(ROOT / "mlops-models/src"), str(ROOT / "mlops-features")]
import mlops_models
import mlops_skeleton
for package in (mlops_models, mlops_skeleton):
    assert Path(package.__file__).resolve().is_relative_to(ROOT.resolve()), (
        f"STOP: this kernel loaded code from {package.__file__}. Restart this notebook's kernel for the new checkout."
    )
from mlops_models.pipelines.source import (
    source_identity, source_location, upload_source, build_definitions, pipeline_fingerprints,
)
from mlops_skeleton.names import discover_models, validate_namespace

validate_namespace(namespace)
available_models = [folder.name for folder in discover_models(ROOT / "mlops-models/src/mlops_models/models")]
model_ids = available_models if model_ids is None else model_ids
assert model_ids and all(model in available_models for model in model_ids), f"STOP: select model folders from {available_models}."
def csv_settings():
    # Recheck if the CSV, model list or any use-case parameters changed.
    return csv_input, tuple(
        (model, (ROOT / "mlops-models/src/mlops_models/models" / model / "parameters.json").read_text())
        for model in model_ids
    )

model_targets = {}
for model in model_ids:
    parameters_path = ROOT / "mlops-models/src/mlops_models/models" / model / "parameters.json"
    parameters = json.loads(parameters_path.read_text())
    for key in ("target_key", "primary_key"):
        assert isinstance(parameters.get(key), str) and parameters[key].strip(), f"STOP: set {key} in {parameters_path}."
    assert parameters.get("task_type") in {"classification", "regression"}, f"STOP: task_type must be classification or regression in {parameters_path}."
    model_targets[model] = parameters["target_key"]
    print("Model:", model, "| target_key:", repr(parameters["target_key"]),
          "| primary_key:", repr(parameters["primary_key"]), "| task_type:", parameters["task_type"])
    print("Parameters:", parameters_path)

for code_file in (ROOT / "mlops-models/src/mlops_models").rglob("*.py"):
    compile(code_file.read_text(), str(code_file), "exec")
print("Model Python syntax checks passed:", model_ids)
pipeline_role_arn = role_arn
job_role_arn = job_role_arn.strip() or role_arn
session = boto3.Session()
sm = session.client("sagemaker")
s3_root = f"s3://{bucket}/{namespace}"
identity = source_identity("direct-deploy", deploy_name)
source_uri = source_location(s3_root, identity)
csv_input_uri, csv_local_path, csv_check = "", None, None
if data_source == "csv_direct":
    if csv_input.startswith("s3://"):
        csv_input_uri = csv_input
    else:
        csv_local_path = Path(csv_input).expanduser()
        if not csv_local_path.is_absolute():
            csv_local_path = ROOT / csv_local_path
        csv_local_path = csv_local_path.resolve()
        print("Resolved local CSV:", csv_local_path)
        assert csv_local_path.is_file(), f"STOP: CSV file does not exist: {csv_local_path}. Nothing will be published."
    try:
        if csv_local_path is not None:
            csv_bytes = csv_local_path.read_bytes()
            csv_input_uri = f"{s3_root}/inputs/csv-direct/{deploy_name}/data.csv"
        else:
            csv_bucket, csv_key = csv_input_uri.removeprefix("s3://").split("/", 1)
            response = session.client("s3").get_object(Bucket=csv_bucket, Key=csv_key)
            with response["Body"] as stream:
                csv_bytes = stream.read()
    except Exception as error:
        raise RuntimeError(f"STOP: cannot read the requested CSV {csv_local_path or csv_input_uri}. Nothing will be published: {error}") from error
    columns = next(csv.reader(io.StringIO(csv_bytes.decode("utf-8-sig"))), [])
    print("CHECKED CSV:", str(csv_local_path or csv_input_uri))
    print("Bytes:", len(csv_bytes), "| Columns:", len(columns))
    print("First 30 column names:", columns[:30])
    print("Model targets from parameters.json:", model_targets)
    assert columns and all(columns), "STOP: CSV has no valid header."
    assert len(set(columns)) == len(columns), "STOP: CSV contains duplicate column names."
    target_presence = {model: target in columns for model, target in model_targets.items()}
    for model, present in target_presence.items():
        print(model, "| target in this CSV:", present, "|", repr(model_targets[model]))
    if not all(target_presence.values()):
        print("This CSV does not contain every configured target. Publication can proceed.")
        print("When training, choose the intended ModelId and its CsvInputUri; that model's code handles its target.")
    preview = pd.read_csv(io.BytesIO(csv_bytes), encoding="utf-8-sig", nrows=5, keep_default_na=False)
    assert len(preview), "STOP: CSV has a header but no data rows."
    display(preview.iloc[:, :10])
    csv_check = {"source": str(csv_local_path or csv_input_uri), "sha256": hashlib.sha256(csv_bytes).hexdigest(),
                 "bytes": len(csv_bytes), "columns": columns, "model_targets": model_targets, "target_presence": target_presence}
    csv_checked_settings = csv_settings()
    csv_preflight_ok = True
    print("CSV FILE CHECK PASSED | SHA-256:", csv_check["sha256"])
    print("Will use S3 CSV:", csv_input_uri)
    del csv_bytes

print("Checkout:", ROOT)
print("Region:", session.region_name)
print("Pipeline role:", pipeline_role_arn)
print("Job role:", job_role_arn)
print("Models:", model_ids)
print("Source upload destination:", source_uri)
print("Input:", csv_input_uri if data_source == "csv_direct" else feature_store_config_uri)
print("Pipelines:", namespace + "-training", "|", namespace + "-inference")
print("Run outputs:", s3_root + "/models/<ModelId>/runs/<execution-id>/")
print("Source identity:", identity)

## Pin the image

- Resolve a supplied image tag to its immutable digest.
- If only a repository is supplied, explicitly report the newest tagged image.
- Reuse that digest for the published definitions.

In [ ]:
if "@sha256:" not in image_uri:
    registry, repository = image_uri.split("/", 1)
    repository, _, tag = repository.partition(":")
    ecr = session.client("ecr")
    if tag:
        detail = ecr.describe_images(
            registryId=registry.split(".")[0], repositoryName=repository,
            imageIds=[{"imageTag": tag}],
        )["imageDetails"][0]
    else:
        # Repository address only: use its newest tagged image,.
        images = []
        for page in ecr.get_paginator("describe_images").paginate(
                registryId=registry.split(".")[0], repositoryName=repository,
                filter={"tagStatus": "TAGGED"}):
            images.extend(page["imageDetails"])
        assert images, "That repository has no tagged images; paste a full image URI instead."
        detail = max(images, key=lambda item: item["imagePushedAt"])
        print("Pinned the newest tagged image:", detail.get("imageTags"))
    image_uri = registry + "/" + repository + "@" + detail["imageDigest"]
print("Image:", image_uri)

## Build the shared pipelines

- Build one training definition and one inference definition.
- Populate both ModelId dropdowns with the selected folders.
- Verify that the notebook can describe each existing Registry group; missing groups or denied reads stop publication.
- Save definitions locally; display cache settings and Registry group names. This read check does not prove job/write permissions.

In [ ]:
if data_source == "csv_direct":
    assert csv_preflight_ok and csv_checked_settings == csv_settings(), "STOP: rerun the CSV check after changing the input, selected models or their targets."

validate_namespace(environment)
registry_groups = {}
for model in model_ids:
    group_name = f"{registry_group_prefix}{model}-sm-{environment}"
    print("Checking existing Registry group:", group_name)
    group = sm.describe_model_package_group(ModelPackageGroupName=group_name)
    registry_groups[model] = group["ModelPackageGroupArn"]
    print("Registry group:", registry_groups[model])
print("Group reads passed for the notebook caller. Pipeline registration and job reads are verified when executed.")

step_cache_revisions = {}
if cache_steps:
    step_cache_revisions = pipeline_fingerprints(session, csv_input_uri, csv_local_path)
print("Compute step cache:", "enabled for 24 hours" if step_cache_revisions else "disabled")
if step_cache_revisions:
    print("Step revisions:", step_cache_revisions)
    print("Training: GetData, FeatureSelection, Preprocess, Train, Evaluate.")
    print("Processing batch inference: Preprocess, Score, Postprocess.")
    print("Registry checks, registration and conditions run normally. Feature Store reads remain uncached.")

definitions = build_definitions(
    namespace, job_role_arn, image_uri, s3_root, source_uri, identity,
    region=session.region_name, models=model_ids, infrastructure=infrastructure, batch_runtime=batch_runtime,
    feature_store_config_uri=feature_store_config_uri, universe_data_root=universe_data_root,
    data_source=data_source, csv_input_uri=csv_input_uri,
    step_cache_revisions=step_cache_revisions,
    model_code_source=model_code_source,
    environment=environment, registry_group_prefix=registry_group_prefix, include_hpo=include_hpo,
)
WORK = ROOT / ".runs/source-deploy" / namespace
WORK.mkdir(parents=True, exist_ok=True)
for kind, body in definitions.items():
    path = WORK / (kind + ".json")
    path.write_text(json.dumps(body, indent=2) + "\n")
    print(namespace + "-" + kind, "→", " → ".join(step["Name"] for step in body["Steps"]))
    print("Definition:", path.resolve())
print("Model Registry: required; training registers pending versions and inference requires approval.")
print("Registry groups:", list(registry_groups.values()))

print("Batch inference runtime:", batch_runtime)
print("Inference code:", definitions["inference"]["Metadata"]["ModelCodeSource"])
if cache_steps and not infrastructure.get("network", {}).get("isolation", False):
    print("Cached Training needs sagemaker:DescribeTrainingJob on the job execution role.")
print("HPO branch:", "included; UseHPO defaults to false" if include_hpo else "hidden; ordinary training only")

## Upload verified input and source

- Upload the selected local CSV, or verify the supplied S3 CSV.
- Compare the uploaded bytes with the preflight checksum.
- Upload the runtime source used by the jobs.

In [ ]:
csv_upload_verified = False
if data_source == "csv_direct":
    assert csv_preflight_ok and csv_checked_settings == csv_settings(), "STOP: rerun the CSV check after changing the input, selected models or their targets."
    csv_bucket, csv_key = csv_input_uri.removeprefix("s3://").split("/", 1)
    try:
        if csv_local_path is not None:
            assert hashlib.sha256(csv_local_path.read_bytes()).hexdigest() == csv_check["sha256"], "STOP: local CSV changed after validation; rerun the notebook."
            kms = infrastructure.get("output_kms_key_arn", "")
            encryption = {"ServerSideEncryption": "aws:kms", "SSEKMSKeyId": kms} if kms else {}
            session.client("s3").upload_file(str(csv_local_path), csv_bucket, csv_key, ExtraArgs=encryption)
        response = session.client("s3").get_object(Bucket=csv_bucket, Key=csv_key)
        with response["Body"] as stream:
            uploaded_sha256 = hashlib.sha256(stream.read()).hexdigest()
        assert uploaded_sha256 == csv_check["sha256"], "S3 CSV differs from the file checked above."
    except Exception as error:
        raise RuntimeError(f"STOP: CSV upload/readback verification failed at {csv_input_uri}. Pipelines will not be published: {error}") from error
    csv_upload_verified = True
    print("S3 CSV VERIFIED:", csv_input_uri)
    print("SHA-256:", uploaded_sha256, "| Columns:", len(csv_check["columns"]))
uploaded_source = upload_source(session, s3_root, identity)
print("Uploaded source:", uploaded_source)
print("GetData / batch Preprocess input:", csv_input_uri if data_source == "csv_direct" else feature_store_config_uri)

## Publish in Studio

- Create or update the same namespace-training and namespace-inference pipelines.
- Verify their saved definitions and roles; no execution is started.
- Save the publication record and print the ModelId choices.

In [ ]:
if data_source == "csv_direct":
    assert csv_upload_verified and csv_checked_settings == csv_settings(), "STOP: complete CSV verification before publishing."
    for body in definitions.values():
        saved_csv = next(p["DefaultValue"] for p in body["Parameters"] if p["Name"] == "CsvInputUri")
        assert saved_csv == csv_input_uri, "STOP: definition uses another CSV; rerun all cells."
published = {}
for kind, body in definitions.items():
    name = f"{namespace}-{kind}"
    request = {
        "PipelineName": name, "RoleArn": pipeline_role_arn,
        "PipelineDefinition": json.dumps(body),
    }
    try:
        existing = sm.describe_pipeline(PipelineName=name)
    except sm.exceptions.ClientError as error:
        if "does not exist" not in error.response["Error"]["Message"]:
            raise
        response = sm.create_pipeline(**request, Tags=[
            {"Key": "Deployment", "Value": namespace},
            {"Key": "SourceDeploy", "Value": identity["mode"]},
        ])
        version = 1  # The initial version of a newly created pipeline.
    else:
        tags = {t["Key"]: t["Value"] for t in sm.list_tags(ResourceArn=existing["PipelineArn"])["Tags"]}
        if tags.get("Deployment") != namespace or tags.get("SourceDeploy") != identity["mode"]:
            raise ValueError(f"{name} belongs to another deployment.")
        response = sm.update_pipeline(**request)
        version = response["PipelineVersionId"]
    actual = sm.describe_pipeline(PipelineName=name, PipelineVersionId=version)
    assert json.loads(actual["PipelineDefinition"]) == body, "Published definition differs; inspect before executing."
    assert actual["RoleArn"] == pipeline_role_arn, "Saved pipeline role differs; inspect before executing."
    published[kind] = {"name": name, "arn": response["PipelineArn"], "version": version}
    print("Published:", name, "| version:", version, "| saved role:", actual["RoleArn"])

publication = {
    "namespace": namespace, "pipelines": published, "models": model_ids,
    "source": identity, "source_uri": source_uri, "image": image_uri,
    "s3_root": s3_root, "batch_runtime": batch_runtime, "region": session.region_name,
    "environment": environment, "registry_group_prefix": registry_group_prefix, "registry_groups": registry_groups, "include_hpo": include_hpo,
    "feature_store_config_uri": feature_store_config_uri, "universe_data_root": universe_data_root,
    "data_source": data_source, "csv_input_uri": csv_input_uri,
    "step_cache_revisions": step_cache_revisions,
    "model_code_source": definitions["inference"]["Metadata"]["ModelCodeSource"],
    "csv_check": csv_check,
}
path = WORK / "publication.json"
path.write_text(json.dumps(publication, indent=2) + "\n")
print("Publication complete:", path.resolve())
print("One training pipeline and one inference pipeline are ready in Studio.")
print("ModelId dropdown:", model_ids)
print("For a new model, complete training and approve its version in Model Registry.")
print("For an already approved model, start inference directly with its ModelId and scoring CsvInputUri.")
if model_code_source == "pipeline":
    print("This publication uses uploaded inference code; no training rerun is required for a compatible saved model.")
print("Inference resolves the highest Completed, Approved version; it has no ModelDataUri override.")
if data_source == "csv_direct":
    print("Use the newly printed pipeline version and this CsvInputUri:", csv_input_uri)
    print("A previous execution/version or a Studio parameter override can select a different CSV.")

## Execute

**Already have an approved model?** Open the printed inference pipeline, execute
its latest version, and select that ModelId and the scoring CsvInputUri. This
notebook defaults to `model_code_source="pipeline"`, so your updated prediction
files run with the existing registered model and metadata. You can proceed
directly to inference; the training steps below are for creating a new candidate.

In SageMaker Studio, open **Pipelines** and select the printed training pipeline.
Create an execution, choose **ModelId**, and check **CsvInputUri**.
HPO is hidden by default. If you enabled include_hpo when publishing, keep **UseHPO=false** for ordinary training.
Training must pass Evaluate and QualityGate, then complete **RegisterModel**.
Registration creates a **PendingManualApproval** version in `<prefix><ModelId>-sm-<environment>`.
RegisterModel is a Processing job: its **job role** needs permission to describe the existing group and create/describe versions. It does not create groups.
In **Models → Registered models**, open that group, inspect the version and change
its approval status to **Approved** using your authorized operator role.

Start the inference pipeline with the same **ModelId** and the scoring **CsvInputUri**.
**ResolveModel** selects the highest Completed, Approved version. It records the
package ARN, image and saved-model S3 URI in the execution's `package.json`.
The remaining steps download that registered artifact and produce
`predictions/predictions.csv` under the inference execution.

A denied Registry call, missing group, or missing approved version fails visibly.
There is no direct model-artifact input or Registry bypass. Registration and
approval lookup run on every execution; matching compute steps can still use caching.

The notebook verifies the CSV file, not its suitability for every model. Each use
case reads its own `target_key`, `primary_key` and `task_type` from `parameters.json`.
Set `task_type` to `classification` or `regression`. The other parameters are passed
through unchanged to the model code. The demo values `target_value` and
`churned_flag` are example target columns, not required framework names. Replace
them with your actual target names. No targets are generated or renamed here.

To restore HPO later, set `include_hpo=True` in this notebook's Defaults cell and republish. The selected models must
have compatible tuning settings, and the image must support HPO startup. This
restores the training-mode selector and tuning inputs; it does not start a job.

To switch models, start a new execution with another ModelId. Set CsvInputUri too
if that model uses different data; execution-time overrides need their own input validation. To add/rename models, edit the folders, restart
the notebook kernel, and run this notebook again. The containing checkout folder may have a different name;
run the notebook from inside that checkout so its ROOT discovery selects these files.